# NB02 · Régression linéaire : comment une machine apprend

**R5-07 · Introduction à l'intelligence artificielle · Séance 2**

*© Romain Dulout, UPPA. Supports de cours originaux : toute reproduction ou réutilisation, même partielle, nécessite l'accord de l'auteur.*

Au NB01, vous avez préparé les données. On entraîne maintenant un **modèle** : une formule qui prend les caractéristiques d'un diamant et qui renvoie un prix estimé.

Le modèle le plus simple est une **droite** : `prix = a × carat + b`. Entraîner le modèle, c'est trouver les meilleures valeurs de `a` et `b`. « Meilleures » veut dire : celles qui font **le moins d'erreurs** sur les diamants dont on connaît déjà le prix.

### Objectifs
À la fin de ce notebook, vous serez capable de :
1. construire un **modèle de référence** pour savoir si un modèle sert à quelque chose ;
2. entraîner une **régression linéaire** avec scikit-learn et lire ses paramètres ;
3. expliquer ce que fait la **descente de gradient**, en l'ayant codée vous-même ;
4. **évaluer** un modèle avec la MAE, la RMSE et le R², et dire ce que chacune mesure ;
5. comparer le score sur l'**entraînement** et sur le **test**.

### Organisation
- Le **socle** (exercices 0 à 6) est faisable par tout le monde. C'est lui qui compte.
- Les **bonus**, notés de ⭐ à ⭐⭐⭐ selon leur difficulté, sont pour celles et ceux qui vont plus vite. Ils ne sont jamais évalués.
- `grader.check(...)` vérifie votre réponse : elle indique si c'est juste, ou donne un indice pour corriger.
- Les indices donnent souvent seulement le **nom d'une fonction** : pour savoir ce qu'elle fait et quels arguments elle attend, placez le curseur sur son nom et appuyez sur **Maj + Tab**, ou exécutez `help(...)`.
- Dans le code, `...` signifie **à compléter**.

### Vocabulaire
| Mot | Sens |
|---|---|
| **features** | les variables d'entrée, ce qu'on connaît du diamant (carat, couleur…), rangées dans `X` |
| **target** (cible) | ce qu'on veut prédire, ici le prix, rangé dans `y` |
| **modèle** | une fonction paramétrée : une formule qui transforme les features en une prédiction de la target |
| **paramètres** | les nombres que la machine estime pendant l'entraînement (ici `a` et `b`) |
| **entraîner** (*fit*) | chercher les paramètres qui font le moins d'erreurs sur les données d'entraînement |
| **prédire** (*predict*) | appliquer la formule à de nouveaux diamants |
| **fonction de coût** | le nombre qui mesure « à quel point le modèle se trompe », noté $J(a, b)$ : ici la MSE, vue comme une fonction des paramètres. Entraîner, c'est la minimiser |

## Exercice 0 · Démarrer

Écrivez **le même prénom qu'au NB01** : vous retrouvez ainsi votre échantillon.

`ct.load_clean_diamonds()` recharge votre échantillon **nettoyé et encodé exactement comme dans la correction du NB01**. Même si votre NB01 n'est pas fini, tout le monde repart de la même base propre.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import otter

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

import course_tools as ct
from course_tools import widgets

grader = otter.Notebook(tests_dir="tests")
sns.set_theme()

ct.start("...")   # ← le même prénom qu'au NB01

In [ ]:
df = ct.load_clean_diamonds()
df.head()

Rappel : `cut`, `color` et `clarity` ont été **encodées** au NB01 (Fair = 0 … Ideal = 4, J = 0 … D = 6, I1 = 0 … IF = 7). Plus le nombre est grand, meilleure est la qualité.

On découpe comme au NB01 : 80 % des diamants pour **entraîner**, 20 % gardés de côté pour **tester**. Le modèle ne verra jamais les diamants du test pendant l'entraînement. Cette cellule est déjà complète, exécutez-la.

In [ ]:
X = df.drop(columns="price")   # les entrées (features)
y = df["price"]                # la cible

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=ct.seed())
print(f"{len(X_train)} diamants pour entraîner, {len(X_test)} pour tester")

## Exercice 1 · Le modèle de référence

**La situation.** Une joaillerie de Biarritz rachète des diamants à des particuliers pour les revendre. Pour fixer son prix de rachat, elle applique une règle simple : elle propose pour chaque diamant **le prix moyen des diamants qu'elle a déjà vendus**, environ 4 000 dollars, quel que soit le diamant.

Résultat : elle perd de l'argent. Elle paie beaucoup trop cher les petits diamants, et les propriétaires de gros diamants, à qui elle propose bien trop peu, vont voir la concurrence.

**Votre mission.** En tant qu'ingénieur ou ingénieure, vous allez construire à partir des données un estimateur de prix fiable. Dans ce notebook, les diamants d'entraînement jouent le rôle de l'historique des ventes de la joaillerie, et les diamants du test celui des nouveaux diamants qu'on lui apporte.

Avant de construire un modèle intelligent, il faut savoir **à quoi le comparer**. La règle actuelle de la joaillerie est le modèle le plus simple possible : prédire **le même prix pour tous les diamants**, le prix moyen de l'entraînement. On l'appelle le **modèle de référence** (*baseline*). C'est lui qu'il faut battre : un vrai modèle n'a d'intérêt que s'il fait **nettement mieux**.

### Combien coûte la règle de la joaillerie ?

L'estimateur de la joaillerie tient en un calcul : additionner les prix de tous les diamants déjà vendus, puis diviser par leur nombre. Le résultat, le prix moyen, sert ensuite d'estimation pour **tous** les nouveaux diamants.

Pour savoir de combien cette règle se trompe, on utilise la **MAE** (*Mean Absolute Error*, erreur absolue moyenne) : pour chaque diamant on calcule l'écart entre le vrai prix et le prix prédit, sans tenir compte du signe, puis on fait la moyenne.

$$\text{MAE} = \text{moyenne}\big(\,|\text{vrai prix} - \text{prix prédit}|\,\big)$$

Elle s'exprime **en dollars** : c'est l'erreur que la joaillerie commet en moyenne sur chaque diamant.

> **Prédire avant d'exécuter** : le prix moyen d'un diamant est d'environ 4 000 dollars. Si on prédit 4 000 dollars pour tout le monde, de combien de dollars se trompe-t-on en moyenne ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
mean_price = y_train.mean()
predictions_baseline = np.full(len(y_test), mean_price)   # le même prix pour chaque diamant du test
print(f"Prix moyen à l'entraînement : {mean_price:.0f} $")

**À vous.** Calculez `mae_baseline`, la MAE du modèle de référence **sur le test**, avec numpy (sans scikit-learn).

Les prédictions de la joaillerie sont déjà prêtes dans `predictions_baseline` (le prix moyen, répété pour chaque diamant du test). Il reste à les comparer aux vrais prix, rangés dans `y_test`, en appliquant la formule de la MAE ci-dessus.

Fonctions utiles : `np.abs` et `np.mean`.

In [ ]:
mae_baseline = ...
print(f"Le modèle de référence se trompe en moyenne de {mae_baseline:.0f} $")

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

In [ ]:
grader.check("q1")

## Pause cours · Ajustez la droite vous-même

Le modèle de référence prédisait le même prix pour tous. On passe maintenant à un modèle qui tient compte du poids : la droite `prix = a × carat + b`. Avant de laisser la machine trouver `a` et `b`, essayez à la main :
- le curseur **pente a** règle l'inclinaison de la droite ;
- le curseur **ordonnée b** la monte ou la descend.

Au-dessus du graphique, deux mesures de l'erreur se mettent à jour :
- la **MSE** (*Mean Squared Error*, erreur quadratique moyenne) est la moyenne des erreurs **au carré**. C'est la fonction de coût que la machine va chercher à rendre la plus petite possible ;
- la **MAE** est la même erreur en dollars, plus facile à lire.

Essayez de descendre sous une MAE de 1 100 dollars, puis cochez la case pour voir la droite trouvée par la machine.

> Et si la machine bougeait les curseurs à votre place ? C'est précisément ce qu'on appelle l'entraînement : c'est l'exercice 2.

In [ ]:
widgets.manual_line_fit(X_train["carat"], y_train)

## Exercice 2 · Régression simple : du carat au prix

La joaillerie le sait bien : plus un diamant est lourd, plus il est cher. Premier vrai modèle, donc : une droite qui estime le prix à partir du carat seul.

Avec scikit-learn, entraîner un modèle se fait toujours en trois temps, quel que soit le modèle :

```python
model = LinearRegression()        # 1. créer le modèle (vide)
model.fit(..., ...)               # 2. l'entraîner sur des entrées et leurs vraies cibles : il cherche les meilleurs paramètres
model.predict(...)                # 3. prédire à partir de nouvelles entrées
```

> **Comment `fit` trouve-t-il `a` et `b` ?** `LinearRegression` utilise la **méthode des moindres carrés** : elle cherche la droite qui rend la plus petite possible la somme des erreurs au carré, c'est-à-dire la MSE. Pour une droite, ce minimum se calcule directement avec une formule, sans tâtonner : `a = cov(carat, prix) / var(carat)`, puis `b = moyenne(prix) - a × moyenne(carat)`. Il n'y a ni point de départ, ni itérations.
>
> **La subtilité** : cette formule n'existe que pour les modèles linéaires. Pour presque tous les autres modèles, et pour tous les réseaux de neurones, on ne sait pas calculer le minimum directement : on s'en approche pas à pas, par **descente de gradient**. À l'exercice 3, vous coderez cette descente sur la même droite, et vous retrouverez les mêmes `a` et `b` que `LinearRegression` : deux chemins différents vers le même minimum.

**Ce qu'on donne au modèle.** `fit` attend deux choses : les entrées, ici **seulement la colonne du carat** des diamants d'entraînement, et les vrais prix de ces mêmes diamants. Attention : scikit-learn veut les entrées sous forme de **tableau** (un DataFrame, même s'il n'a qu'une colonne), pas sous forme de série. Pour garder un tableau, on sélectionne la colonne avec **deux** paires de crochets.

**Ce que le modèle apprend.** Une droite `prix = a × carat + b` est entièrement décrite par deux nombres :
- la **pente** `a` (*slope* en anglais) : de combien de dollars le prix augmente quand le diamant pèse 1 carat de plus. Une pente de 5 000 voudrait dire « 5 000 dollars de plus par carat » ;
- l'**ordonnée à l'origine** `b` (*intercept*) : le prix prédit pour un diamant de 0 carat, c'est-à-dire la hauteur à laquelle la droite coupe l'axe vertical.

Une fois entraîné, le modèle range la pente dans son attribut `coef_` (un tableau, avec un nombre par variable d'entrée : ici, il n'y en a qu'une) et l'ordonnée à l'origine dans `intercept_`.

**À vous.** Créez `simple_model`, entraînez-le sur le carat uniquement, puis rangez sa pente dans `slope` et son ordonnée à l'origine dans `intercept`.

In [ ]:
# Créez le modèle (rangé dans simple_model), puis entraînez-le sur le carat seul
...

slope = ...       # la pente a
intercept = ...   # l'ordonnée à l'origine b
print(f"prix = {slope:.0f} × carat {intercept:+.0f}")

> **Prédire avant d'exécuter** : d'après ce modèle, quel est le prix d'un diamant de **0 carat** ? Est-ce que ça a du sens ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

**À vous.** Calculez `price_0_carat`, le prix prédit par le modèle pour un diamant de 0 carat (avec la formule de la droite et les deux paramètres que vous venez de récupérer).

In [ ]:
price_0_carat = ...
print(f"Prix d'un diamant de 0 carat selon le modèle : {price_0_carat:.0f} $")

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

Le prix prédit est négatif. Le modèle n'a jamais vu de diamant de moins de 0,2 carat : en dehors de la plage des données d'entraînement, une formule peut dire n'importe quoi. On dit qu'on **extrapole**.

**À vous.** Tracez sur un même graphique :
1. le **nuage de points** des diamants du test, avec le carat en abscisse et le vrai prix en ordonnée (`plt.scatter`) ;
2. la **droite** du modèle, de 0 carat jusqu'au plus gros diamant (`plt.plot`). Les carats de la droite sont déjà préparés dans `xs` ; les prix correspondants s'obtiennent avec `simple_model.predict`.

Donnez à chaque tracé un `label`, pour la légende. Puis observez : en dessous de quel poids la droite passe-t-elle sous 0 ? Le nuage de points a-t-il vraiment la forme d'une droite ?

**Mes observations :** *(double-cliquez ici)*

In [ ]:
xs = pd.DataFrame({"carat": np.linspace(0, X["carat"].max(), 100)})   # 100 carats régulièrement espacés, de 0 au plus gros diamant

plt.figure(figsize=(8, 4.5))
...   # 1. le nuage de points des diamants du test
...   # 2. la droite du modèle
plt.axhline(0, color="grey", lw=1)   # la ligne des 0 dollars
plt.xlabel("carat")
plt.ylabel("prix ($)")
plt.legend()
plt.show()

**À vous.** Calculez `mae_simple`, la MAE de `simple_model` sur le test. Cette fois, utilisez la fonction de scikit-learn `mean_absolute_error`, importée au début (Maj + Tab pour voir ses arguments). Fait-on mieux que la règle de la joaillerie ?

In [ ]:
predictions_simple = simple_model.predict(X_test[["carat"]])
mae_simple = ...
print(f"Modèle de référence : {mae_baseline:.0f} $   vs   simple_model : {mae_simple:.0f} $")

In [ ]:
grader.check("q2")

## Exercice 3 · La descente de gradient, à la main

À l'exercice 2, `fit` a trouvé `a` et `b` d'un coup, grâce à la formule des moindres carrés. Pourquoi alors coder autre chose ? Parce que cette formule n'existe que pour les modèles linéaires. Les bibliothèques de réseaux de neurones comme TensorFlow ou PyTorch n'en ont pas : elles entraînent **tous** leurs modèles par **descente de gradient**, la méthode vue en séance. Vous allez la coder vous-même sur la même droite, et vérifier qu'elle retrouve les `a` et `b` de scikit-learn.

**La fonction de coût.** C'est la MSE du cours, vue comme une fonction des deux paramètres :

$$J(a, b) = \frac{1}{n} \sum_{i=1}^{n} \big(y_i - \hat{y}_i\big)^2 \qquad \text{avec} \qquad \hat{y}_i = a\,x_i + b$$

où $n$ est le nombre de diamants d'entraînement, $x_i$ le carat du diamant $i$, $y_i$ son vrai prix et $\hat{y}_i$ le prix prédit.

**L'algorithme.** On part de $a = b = 0$, puis on répète :
1. calculer les prix prédits ;
2. calculer le **gradient** $\big(\partial J / \partial a,\ \partial J / \partial b\big)$, avec les dérivées partielles de $J$, que vous calculez juste en dessous ;
3. faire un pas dans la direction **opposée** : $a \leftarrow a - \eta\, \partial J / \partial a$, et de même pour $b$.

La taille des pas est réglée par le **taux d'apprentissage** $\eta$ (*learning rate*), écrit `eta` dans le code.

> **Rappel numpy.** $\frac{1}{n} \sum_{i}$ s'écrit `np.mean(...)` : une somme divisée par le nombre de termes est une moyenne. Les opérations entre tableaux numpy se font élément par élément, sans boucle.

**À vous, sur papier.** Dérivez $J(a, b)$ pour obtenir les deux composantes du gradient, $\partial J / \partial a$ et $\partial J / \partial b$. C'est le calcul fait en séance : si vous l'avez déjà, recopiez votre résultat.

Indice : la dérivée de $u^2$ est $2\,u\,u'$, et $\hat{y}_i = a\,x_i + b$ dépend de $a$ et de $b$.

**Mes dérivées :** *(double-cliquez ici)*

**À vous, en code.** Complétez tout l'intérieur de la boucle : les prix prédits, la MSE ajoutée à `history`, vos deux dérivées traduites en numpy (`slope_a` et `slope_b`), puis la mise à jour de `a` et de `b`. Gardez `eta = 0.1` et 3 000 itérations.

In [ ]:
carat = X_train["carat"].to_numpy()       # les x_i
true_price = y_train.to_numpy()           # les y_i

a, b = 0.0, 0.0        # le point de départ
eta = 0.1              # η, le taux d'apprentissage
history = []           # la MSE à chaque itération, pour tracer la descente

for iteration in range(3000):
    predicted_price = ...      # les prix prédits par la droite actuelle
    history.append(...)        # ajouter J(a, b), la MSE actuelle, à l'historique

    slope_a = ...      # ∂J/∂a
    slope_b = ...      # ∂J/∂b

    a = ...
    b = ...

a_gd, b_gd = a, b
print(f"Descente de gradient : prix = {a_gd:.0f} × carat {b_gd:+.0f}")
print(f"scikit-learn         : prix = {slope:.0f} × carat {intercept:+.0f}")

In [ ]:
grader.check("q3")

Comment l'erreur a-t-elle baissé au fil des itérations ? Exécutez la cellule suivante.

> **Vocabulaire.** Dans votre boucle, chaque itération utilise **tous** les diamants d'entraînement pour calculer le gradient : une itération correspond donc à une **époque** (*epoch*), c'est-à-dire un passage complet sur les données. Avec des millions d'exemples, ce calcul devient trop lourd : on calcule alors le gradient sur de petits paquets d'exemples (*mini-batch*), et une époque contient plusieurs pas. Vous le retrouverez avec les réseaux de neurones.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history)
plt.yscale("log")
plt.xlabel("itération")
plt.ylabel("MSE (échelle log)")
plt.title("La descente : l'erreur à chaque pas")
plt.show()

almost_at_min = next(i for i, mse in enumerate(history) if mse <= history[-1] * 1.01)
print(f"Après {almost_at_min} itérations, on est à moins de 1 % de l'erreur finale.")

### Et si on change η ?

Ce widget refait la même descente, sur la surface $J(a, b)$ en 3D : c'est celui de la démo. Choisissez η, puis faites avancer le curseur des itérations. Le titre indique combien d'itérations il faut pour atteindre le minimum (à 1 % près).

- Essayez η = 0.001, puis 0.01, puis 0.1 : comment évolue le nombre d'itérations ?
- Essayez 0.5, 0.55, puis 0.6 : que se passe-t-il ?

> **Prédire avant d'exécuter** : avec η = 0.1, la courbe ci-dessus donne le nombre d'itérations nécessaires. Avec η = 0.001, des pas 100 fois plus petits, combien en faudra-t-il ? Et avec η = 0.55, à peine plus que 0.5 ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
widgets.gradient_descent(X_train["carat"], y_train)

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

> **À retenir.** Trop petit, l'apprentissage est interminable ; trop grand, chaque pas dépasse le minimum et l'erreur explose (on dit que ça **diverge**). Le taux d'apprentissage η n'est pas appris : c'est vous qui le choisissez, on l'appelle un **hyperparamètre**. C'est l'un des réglages les plus importants des réseaux de neurones, que l'on verra plus tard dans le module.

## Exercice 4 · Régression multiple : toutes les variables

Le prix ne dépend pas que du poids : la couleur, la pureté, la taille comptent aussi. Une **régression multiple** utilise toutes les variables à la fois :

$$\text{prix} = a_1 \times \text{carat} + a_2 \times \text{cut} + a_3 \times \text{color} + \dots + b$$

Chaque variable a son propre coefficient, plus un seul $b$ : 9 coefficients et $b$, soit 10 paramètres (en écriture vectorielle, comme dans les slides : $\text{prix} = \mathbf{a} \cdot \mathbf{x} + b$). Le code ne change presque pas par rapport à l'exercice 2 : seules les entrées données au modèle changent.

> **Prédire avant d'exécuter** : `multiple_model`, entraîné sur toutes les variables, fera-t-il mieux que `simple_model` ? Si oui, de combien de dollars environ ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

**À vous.** Entraînez `multiple_model` sur **toutes** les variables, prédisez les prix du test, puis calculez `mae_multiple`.

In [ ]:
# Créez et entraînez multiple_model sur toutes les variables,
# rangez ses prédictions sur le test dans predictions, puis sa MAE dans mae_multiple
...

print(f"Modèle de référence : {mae_baseline:.0f} $   vs   simple_model : {mae_simple:.0f} $   vs   multiple_model : {mae_multiple:.0f} $")

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

In [ ]:
grader.check("q4")

**Que dit chaque coefficient ?** `multiple_model` a appris un coefficient par variable. Ils sont rangés dans `multiple_model.coef_`, un tableau de 9 nombres dans l'ordre des colonnes de `X_train`. Ce tableau seul est difficile à lire : la cellule suivante met chaque coefficient en face du nom de sa colonne (c'est le rôle de `pd.Series`), les arrondit au dollar, puis les trie du plus petit au plus grand. Exécutez-la.

Un coefficient se lit ainsi : « si cette variable augmente de 1 et que **toutes les autres restent identiques**, le prix prédit change de tant de dollars ». Par exemple, un coefficient de 500 pour `color` voudrait dire : un cran de couleur en plus, à poids, taille et pureté égaux, ajoute 500 dollars au prix prédit.

In [ ]:
coefficients = pd.Series(multiple_model.coef_, index=X_train.columns)   # chaque coefficient en face du nom de sa colonne
coefficients.round(0).sort_values()                                       # arrondis au dollar, du plus petit au plus grand

> **Attention aux interprétations hâtives.** Regardez le coefficient de `x`, la longueur du diamant en millimètres : il est très probablement négatif. Un diamant plus long serait-il moins cher ? Non.
>
> **D'où vient ce signe ?** `carat`, `x`, `y` et `z` mesurent presque la même chose : la taille du diamant. Quand le poids augmente, les trois dimensions augmentent avec lui. Le modèle a donc quatre colonnes pour une seule information, et beaucoup de combinaisons de coefficients donnent presque les mêmes prédictions : tout mettre sur `carat`, ou en mettre un peu plus sur `carat` et en retirer sur `x`… Il en choisit une un peu au hasard. Pris seul, le coefficient de `x` ne veut alors plus rien dire.
>
> Ce phénomène s'appelle la **multicolinéarité** : des variables d'entrée très liées entre elles. Les prédictions restent bonnes, mais les coefficients de ces variables ne se lisent plus un par un. Ceux de `cut`, `color` et `clarity`, qui ne mesurent pas la taille, restent lisibles : ils sont positifs, comme on s'y attend.

## Exercice 5 · Trois façons de mesurer l'erreur

| Métrique | Formule | Comment la lire |
|---|---|---|
| **MAE** | moyenne des valeurs absolues des erreurs | erreur moyenne en dollars. Simple à expliquer à un client |
| **RMSE** | racine carrée de la moyenne des erreurs au carré (la racine de la MSE) | aussi en dollars, mais **les grosses erreurs comptent beaucoup plus** (elles sont mises au carré) |
| **R²** | 1 - MSE du modèle / MSE obtenue en prédisant la moyenne pour tous | sans unité. 1 = parfait, 0 = pas mieux que la moyenne, négatif = pire que la moyenne |

Le R² répond à la question : « quelle part des différences de prix entre diamants le modèle explique-t-il ? ».

**À vous.** Calculez les trois métriques de `multiple_model` sur le test, et rangez-les dans `mae`, `rmse` et `r2`. Deux chemins possibles, au choix :
- **avec scikit-learn** : `mean_absolute_error`, `root_mean_squared_error` et `r2_score`, importées au début, s'utilisent toutes de la même façon, avec les vrais prix d'abord et les prix prédits ensuite ;
- **à la main avec numpy**, en traduisant les formules du tableau (`np.mean`, `np.abs`, `np.sqrt`). Pour le R², la moyenne à prédire pour tous est celle des vrais prix du test.

Les deux chemins donnent les mêmes nombres : si vous avez le temps, faites les deux pour vérifier.

In [ ]:
mae = ...
rmse = ...
r2 = ...
print(f"MAE = {mae:.0f} $   ·   RMSE = {rmse:.0f} $   ·   R² = {r2:.3f}")

**Question.** La RMSE est plus grande que la MAE. Pourquoi ? Répondez par une lettre.

- **A.** La RMSE compte les erreurs dans les deux sens, alors que la MAE ne garde que les erreurs positives.
- **B.** Quelques diamants sont très mal prédits, et la RMSE donne beaucoup plus de poids à ces grosses erreurs.
- **C.** La RMSE est exprimée en dollars au carré, donc elle est forcément plus grande.
- **D.** La RMSE pénalise davantage les petites erreurs, qui sont de loin les plus nombreuses.

In [ ]:
answer_rmse = "..."   # "A", "B", "C" ou "D"

In [ ]:
grader.check("q5")

> **Prédire avant d'exécuter** : quel est le R² du **modèle de référence** (celui qui prédit toujours le prix moyen) ?
>
> **Ma prédiction :** *(double-cliquez ici et écrivez votre réponse AVANT d'exécuter la cellule suivante)*

In [ ]:
print(f"R² du modèle de référence : {r2_score(y_test, predictions_baseline):.3f}")

> **Confronter** : votre prédiction était-elle juste ? Si non, expliquez l'écart en une phrase.
>
> **Mon explication :** *(double-cliquez ici)*

## Exercice 6 · Entraînement contre test

Un modèle peut-il « apprendre par cœur » ses données d'entraînement ? Pour le savoir, on compare son score sur les diamants **qu'il a vus** (entraînement) et sur ceux **qu'il n'a jamais vus** (test).

- Si les deux scores sont proches, le modèle **généralise** : il a appris une règle valable pour de nouveaux diamants.
- Si le score d'entraînement est bien meilleur que celui du test, il a appris par cœur des détails propres à l'entraînement : c'est le **surapprentissage** (*overfitting*). On l'étudie dans le NB03.

**À vous.** Calculez le R² de `multiple_model` sur l'entraînement (`r2_train`) et sur le test (`r2_test`).

In [ ]:
r2_train = ...
r2_test = ...
print(f"R² entraînement = {r2_train:.3f}   ·   R² test = {r2_test:.3f}")

**Question.** Que peut-on conclure ? Répondez par une lettre.

- **A.** Le modèle surapprend : le R² d'entraînement est plus élevé que celui du test.
- **B.** Le modèle est parfait : il obtient le même score sur les deux jeux.
- **C.** Le modèle généralise bien : il fait à peu près aussi bien sur des diamants jamais vus.
- **D.** On ne peut rien conclure sans connaître la MAE.

In [ ]:
answer_generalization = "..."   # "A", "B", "C" ou "D"

In [ ]:
grader.check("q6")

### Bilan du socle

| Modèle | MAE sur le test |
|---|---|
| référence (le prix moyen) | ≈ 3 000 dollars |
| `simple_model` (carat seul) | ≈ 1 000 dollars |
| `multiple_model` (toutes les variables) | ≈ 900 dollars |

Une droite et quelques lignes de code divisent l'erreur par plus de trois. La suite est en bonus.

## ⭐ Bonus 1 · Les résidus, et l'astuce du logarithme

> **Ce que vous travaillez.** Diagnostiquer un modèle au-delà d'un seul score, puis corriger sa forme en transformant la cible.
>
> **Le problème à résoudre.** `multiple_model` se trompe beaucoup plus sur les diamants chers que sur les petits, et il prédit parfois des prix négatifs. Une MAE moyenne ne le montre pas.

Le bonus se fait en trois temps : calculer et tracer les résidus de `multiple_model`, lire le graphique, puis corriger le modèle en prédisant le logarithme du prix.

### 1. Les résidus

Un **résidu** est l'erreur sur un diamant : `vrai prix - prix prédit`. Si le modèle était bon partout, les résidus seraient répartis au hasard autour de 0, sans forme particulière.

**L'analyse des résidus, scientifiquement.** Une régression linéaire repose sur une hypothèse : la vraie relation est une droite, et ce que le modèle n'explique pas n'est que du bruit, centré sur 0, de même ampleur partout et sans lien avec la prédiction. Le graphique des résidus vérifie cette hypothèse d'un coup d'œil. S'il a une forme (une courbe, un entonnoir), c'est qu'une partie de l'information est restée dans les erreurs : le modèle peut encore progresser, non pas avec plus de données, mais en changeant de forme. C'est le premier diagnostic de toute régression, en statistique comme en machine learning.

**À vous.** Calculez `residuals`, le résidu de chaque diamant du test. Les vrais prix sont dans `y_test`, les prix prédits par `multiple_model` dans `predictions`.

In [ ]:
residuals = ...

**À vous.** Tracez les résidus : un point par diamant du test, avec le prix prédit en abscisse et le résidu en ordonnée (`plt.scatter`). La ligne du 0, celle d'une prédiction parfaite, est déjà tracée.

In [ ]:
plt.figure(figsize=(8, 4.5))
...   # le nuage des résidus
plt.axhline(0, color="#A8453C", lw=2)   # la ligne du 0 : aucune erreur
plt.xlabel("prix prédit ($)")
plt.ylabel("résidu = vrai - prédit ($)")
plt.show()

### 2. Lire le graphique

**À vous.**
1. Décrivez la forme du nuage en une phrase : les erreurs ont-elles la même taille pour les petits et pour les gros prix ? Le nuage est-il plat, ou courbé ?
2. Regardez la gauche du graphique : certains prix prédits sont **négatifs**, ce qui est impossible pour un diamant. Calculez `n_negative_prices`, le nombre de diamants du test concernés.

**Ma description :** *(double-cliquez ici)*

In [ ]:
n_negative_prices = ...

### 3. La correction : prédire le logarithme du prix

**Le constat.** Les erreurs grossissent avec le prix, et le modèle prédit des prix négatifs pour les petits diamants. La raison : sur un diamant, chaque caractéristique fait gagner un **pourcentage** du prix, pas un nombre fixe de dollars. Or une régression linéaire ajoute toujours le même montant.

**Le modèle qu'on voudrait.** Avec une seule variable pour commencer : un prix qui est **multiplié** par le même facteur à chaque carat en plus.

$$\text{prix} = B \times e^{\,a \times \text{carat}}$$

**Le calcul.** On prend le logarithme (népérien, `np.log` en numpy) des deux côtés, avec $\log(B \times e^{u}) = \log(B) + u$ :

$$\log(\text{prix}) = a \times \text{carat} + \log(B)$$

C'est une **droite** entre le carat et le logarithme du prix. On peut donc la faire apprendre par `LinearRegression`, à condition de lui donner $\log(\text{prix})$ comme cible au lieu du prix : elle trouve $a$ et $b = \log(B)$. Pour revenir aux dollars, on fait le chemin inverse : $\text{prix} = e^{\,a \times \text{carat} + b}$.

**Ce que ça donne.** Sur les diamants, on trouve environ $\text{prix} \approx 510 \times e^{1{,}9 \times \text{carat}}$ : chaque dixième de carat en plus multiplie le prix par 1,2 environ (+20 %), que le diamant soit petit ou gros. Et une exponentielle étant toujours positive, il n'y a plus de prix négatif.

**Avec toutes les variables**, c'est la même chose : $\log(\text{prix}) = a_1 \times \text{carat} + a_2 \times \text{cut} + \dots + b$. On appelle cela un modèle **log-linéaire**, la méthode standard pour prédire des prix ou des salaires.

**À vous**, en quatre étapes :
1. entraînez `log_model` sur `X_train` et le logarithme des prix d'entraînement (`np.log`) ;
2. prédisez sur `X_test` : vous obtenez des logarithmes de prix ;
3. repassez en dollars avec `np.exp`, et rangez ces prix dans `price_predictions` ;
4. calculez `mae_log`, la MAE en dollars sur le test, pour la comparer à celle de `multiple_model`.

In [ ]:
log_model = LinearRegression()

# Étape 1 : entraîner log_model sur le logarithme des prix d'entraînement


# Étape 2 : prédire sur le test (des logarithmes de prix)


# Étape 3 : repasser en dollars, dans price_predictions


# Étape 4 : la MAE en dollars sur le test
mae_log = ...
print(f"multiple : {mae_multiple:.0f} $   ·   multiple sur log(prix) : {mae_log:.0f} $")

In [ ]:
grader.check("bonus1")

**Le graphique des résidus, à nouveau.** La cellule suivante retrace les résidus, cette fois avec les prix prédits par `log_model`. Comparez avec le premier graphique : que sont devenus les prix négatifs ? Et les erreurs sur les petits diamants, puis sur les plus chers ?

**Mes observations :** *(double-cliquez ici)*

In [ ]:
residuals_log = y_test - price_predictions
plt.figure(figsize=(8, 4.5))
plt.scatter(price_predictions, residuals_log, s=8, alpha=0.3)
plt.axhline(0, color="#A8453C", lw=2)   # la ligne du 0 : aucune erreur
plt.xlabel("prix prédit par log_model ($)")
plt.ylabel("résidu = vrai - prédit ($)")
plt.show()

## ⭐⭐ Bonus 2 · Prédire le prix des maisons d'Ames

> **Ce que vous travaillez.** Refaire seul ou seule une démarche complète (choix des variables, valeurs manquantes, découpage, modèle de référence, régression) sur des données que vous ne connaissez pas.
>
> **Le problème à résoudre.** Dans un vrai projet, personne ne vous dit quelles colonnes prendre ni comment traiter les trous : la méthode doit se transférer d'un jeu de données à l'autre.

On quitte les diamants pour l'immobilier. Ames est une ville universitaire de l'Iowa, au centre des États-Unis. Le jeu de données *Ames Housing* décrit 2 930 maisons qui y ont été vendues entre 2006 et 2010, avec 82 colonnes. La cible est `SalePrice`, le prix de vente en dollars. Cette fois, **pas de guide** : refaites seul ou seule la démarche de ce notebook.

Quelques colonnes numériques utiles pour commencer :

| Colonne | Signification |
|---|---|
| `Gr Liv Area` | surface habitable hors sous-sol (pieds carrés, 1 pied² ≈ 0,093 m²) |
| `Overall Qual` | qualité générale des matériaux et finitions, de 1 à 10 |
| `Year Built` | année de construction |
| `Total Bsmt SF` | surface du sous-sol (pieds carrés) |
| `Garage Cars` | nombre de voitures dans le garage |

**Votre mission.**
1. Choisissez quelques colonnes numériques, rangez leurs noms dans une liste `columns`, et gérez les valeurs manquantes (il y en a quelques-unes).
2. Découpez en entraînement / test avec `random_state=ct.seed()`, en nommant les résultats `Xa_train`, `Xa_test`, `ya_train` et `ya_test`.
3. Exécutez la cellule du modèle de référence, fournie : c'est le score à battre.
4. Entraînez une régression linéaire et calculez `mae_ames`, sa MAE sur le test.

In [ ]:
ames = ct.load_ames()
ames.head()

In [ ]:
# 1. Les colonnes (dans columns) et les valeurs manquantes


# 2. Le découpage : Xa_train, Xa_test, ya_train, ya_test


In [ ]:
# Le modèle de référence, fourni : le prix moyen des maisons d'entraînement, prédit pour toutes les maisons du test
mae_ames_baseline = mean_absolute_error(ya_test, np.full(len(ya_test), ya_train.mean()))
print(f"Modèle de référence : {mae_ames_baseline:.0f} $")

In [ ]:
# 4. Votre régression linéaire, et sa MAE sur le test

mae_ames = ...
print(f"Modèle de référence : {mae_ames_baseline:.0f} $   vs   votre régression : {mae_ames:.0f} $")

In [ ]:
grader.check("bonus2")

**Bonus du bonus.** Quelle variable fait le plus gagner ? Deux façons de répondre, à compléter dans la boucle ci-dessous :
- entraîner un modèle avec **cette seule variable** : plus sa MAE est basse, plus la variable porte d'information à elle seule ;
- entraîner le modèle complet **sans cette variable** : plus la MAE remonte, plus elle était indispensable.

Et le logarithme du prix, vu au bonus 1, aide-t-il aussi pour les maisons ?

In [ ]:
for column in columns:
    # a. un modèle avec cette seule variable, et sa MAE sur le test
    mae_alone = ...

    # b. le modèle avec toutes les autres variables, et sa MAE sur le test
    mae_without = ...

    print(f"{column:15}  seule : {mae_alone:7.0f} $   ·   sans elle : {mae_without:7.0f} $")

## ⭐⭐⭐ Bonus 3 · Divisez l'erreur par deux

> **Ce que vous travaillez.** L'ingénierie des variables (*feature engineering*) : transformer, créer ou retirer des colonnes pour que le modèle voie mieux la structure des données.
>
> **Le problème à résoudre.** Une régression linéaire ne sait tracer que des droites (des plans, avec plusieurs variables). Or la vraie relation est courbe. Plutôt que de changer d'algorithme, on change ce qu'on lui donne en entrée.

Retour aux diamants, avec `X_train` et `y_train`. `multiple_model` (exercice 4) se trompe d'environ 900 dollars en moyenne. Le défi : construire un modèle dont la MAE sur le test est **au plus la moitié** de `mae_multiple`, en restant avec `LinearRegression` (pas d'autre algorithme).

Tout le reste est permis : transformer la cible, transformer les variables, en créer de nouvelles, en enlever… Une seule règle : le modèle ne s'entraîne **que** sur `X_train` et `y_train`.

**Indices**, à lire un par un, seulement si vous êtes bloqué ou bloquée.

**Indice 1.** Commencez par le bonus 1 : le logarithme du prix fait déjà gagner beaucoup.

**Indice 2.** Le prix est à peu près proportionnel à une **puissance** du carat. Si on prend le log du prix, que faudrait-il faire au carat pour que la relation devienne une droite ? Et les dimensions `x`, `y`, `z`, qui disent presque la même chose que le carat ?

*Pour aller plus loin, une fois trouvé :* si `prix ≈ k × carat^p` (une loi de puissance), alors `log(prix) = log(k) + p × log(carat)`, qui est une droite. C'est le modèle **log-log**, classique en économie : le coefficient `p` s'appelle une **élasticité** et se lit « 1 % de carat en plus donne environ p % de prix en plus ».

**Indice 3.** L'encodage ordinal suppose que passer de J à I fait gagner autant que passer de E à D. Est-ce vrai ? `pd.get_dummies(X, columns=["color", "clarity"])` crée une colonne 0/1 par catégorie (encodage *one-hot*, vu au NB01), et laisse le modèle apprendre un effet différent pour chacune. C'est la façon standard de donner des catégories à un modèle linéaire, quand rien ne garantit que les écarts entre catégories sont réguliers.

> **Règle d'honneur.** Si vous essayez vingt idées en regardant à chaque fois le score sur le test, vous finissez par choisir celle qui a de la chance sur *ce* test : le test n'est plus vraiment « jamais vu ». Au NB03, vous verrez comment l'éviter avec un jeu de **validation**.

In [ ]:
# Votre modèle ici. À la fin, predictions_bonus doit contenir le prix prédit (en dollars)
# pour chaque diamant de X_test, dans le même ordre.

predictions_bonus = ...
print(f"MAE du bonus 3 : {mean_absolute_error(y_test, predictions_bonus):.0f} $   (objectif : {mae_multiple / 2:.0f} $ ou moins)")

In [ ]:
grader.check("bonus3")